# AskSight — QLoRA Fine-tuning: Qwen3-VL-4B-Instruct

Fine-tunes `Qwen/Qwen3-VL-4B-Instruct` using QLoRA (4-bit quantization + LoRA adapters)  
on **VizWiz-VQA**, **TextVQA**, and **DocVQA** for blind and low-vision assistive use.

**Run on:** Kaggle (free T4/P100 GPU, 30 hrs/week) or Google Colab (T4 GPU)

**What you get after training:**
- A `checkpoints/asksight-v1/` folder with LoRA adapter weights (~50 MB)
- Set `ASKSIGHT_MODEL_CHECKPOINT=checkpoints/asksight-v1` in backend `.env` to use it

**Estimated time:**
| Dataset size | Epochs | GPU | Time |
|---|---|---|---|
| 5 000 samples | 3 | Kaggle T4 | ~2 hrs |
| 15 000 samples | 3 | Kaggle T4 | ~6 hrs |
| 20 000 samples | 3 | Kaggle P100 | ~8 hrs |

## 0. Configuration
Edit these values before running. Everything else is automatic.

In [ ]:
import os
from pathlib import Path

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen3-VL-4B-Instruct"
OUTPUT_DIR      = "./checkpoints/asksight-v1"

# ── Dataset paths (JSONL with 'image_path', 'question', 'answer' per line) ─
# Point DATA_DIR at the folder containing the dataset subfolders.
# Kaggle: /kaggle/input   Colab: /content/data or a Drive folder
DATA_DIR        = Path(os.environ.get("ASKSIGHT_DATA_DIR", "/kaggle/input"))

def find_jsonl(env_name, *relative_paths):
    """Use an explicit env override, else the first matching file under DATA_DIR."""
    if os.environ.get(env_name):
        return os.environ[env_name]
    for rel in relative_paths:
        path = DATA_DIR / rel
        if path.is_file():
            return str(path)
    return str(DATA_DIR / relative_paths[0])

VIZWIZ_PATH     = find_jsonl("VIZWIZ_PATH", "vizwiz-vqa/train.jsonl", "vizwiz/train.jsonl")
TEXTVQA_PATH    = find_jsonl("TEXTVQA_PATH", "textvqa/train.jsonl")
DOCVQA_PATH     = find_jsonl("DOCVQA_PATH", "docvqa/train.jsonl")

# ── QLoRA hyperparameters ─────────────────────────────────────────────────
LORA_RANK       = 16       # higher = more capacity, more VRAM
LORA_ALPHA      = 32       # usually 2x rank
LORA_DROPOUT    = 0.05

# ── Training ──────────────────────────────────────────────────────────────
EPOCHS          = 3
BATCH_SIZE      = 2        # keep low — VLM inputs are large
GRAD_ACCUM      = 8        # effective batch = 2 × 8 = 16
LEARNING_RATE   = 2e-4
MAX_SAMPLES     = 7500     # Sized for Kaggle ~6-hour session (~5.5 hrs runtime)
SEED            = 42

## 1. Install dependencies

In [ ]:
!pip install -q \
    transformers>=4.51.0 \
    peft>=0.14.0 \
    bitsandbytes>=0.45.0 \
    accelerate>=1.3.0 \
    trl>=0.16.0 \
    qwen-vl-utils>=0.0.10 \
    pillow \
    opencv-python-headless

print("✅ Dependencies installed")

## 2. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ No GPU detected. Switch to a GPU runtime."

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1e9
print(f"✅ GPU : {gpu.name}")
print(f"   VRAM: {vram_gb:.1f} GB")

if vram_gb < 14:
    print("⚠️  Less than 14 GB VRAM — reduce BATCH_SIZE to 1 if you get OOM errors.")

## 3. Load datasets

Expected JSONL format per line:
```json
{"image_path": "/path/to/image.jpg", "question": "What color is the sign?", "answer": "Red."}
```
VizWiz also supports `"unanswerable": true` — the dataset class handles this automatically.

In [ ]:
import json
from pathlib import Path
from collections import Counter

def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

samples = []

print(f"Data dir: {DATA_DIR}")
for path, tag in [(VIZWIZ_PATH, "vizwiz"), (TEXTVQA_PATH, "textvqa"), (DOCVQA_PATH, "docvqa")]:
    if path and Path(path).exists():
        rows = load_jsonl(path)
        for r in rows:
            r["source"] = tag
        samples.extend(rows)
        print(f"  {tag:10s}: {len(rows):>6,} samples")
    elif path:
        print(f"  ⚠️  {tag}: path not found — {path}")

assert samples, (
    "❌ No samples loaded. Set ASKSIGHT_DATA_DIR (or the *_PATH variables) in Section 0.\n"
    f"   Data dir: {DATA_DIR}\n"
    "   Expected: <dataset>/train.jsonl with image_path, question, answer per line."
)

if MAX_SAMPLES and len(samples) > MAX_SAMPLES:
    import random
    random.seed(SEED)
    random.shuffle(samples)
    samples = samples[:MAX_SAMPLES]
    print(f"  → Capped to {MAX_SAMPLES:,} samples (MAX_SAMPLES)")

source_counts = Counter(s["source"] for s in samples)
print(f"\n✅ Total: {len(samples):,} samples")
print("   Mix  :", dict(source_counts))

## 4. Load model + processor (4-bit QLoRA)

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"Loading processor from {MODEL_ID} ...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
# Set pixel bounds on image_processor directly (avoids FutureWarning in transformers v5+)
processor.image_processor.min_pixels = 256 * 32 * 32
processor.image_processor.max_pixels = 1024 * 32 * 32
print("✅ Processor loaded")

print(f"Loading model in 4-bit from {MODEL_ID} ...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,   # extra memory saving
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("✅ Model loaded")

# Freeze vision encoder — only language decoder LoRA layers train
if hasattr(model, "visual") and model.visual is not None:
    for param in model.visual.parameters():
        param.requires_grad = False
    print("✅ Vision encoder frozen")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect: ~10M trainable / 4B total (~0.25%)

## 5. Dataset class

In [ ]:
import re
from torch.utils.data import Dataset
from PIL import Image

DESCRIBE_PROMPT = (
    "You are a visual assistant for a person with low vision. "
    "Describe this image in 2 to 4 short, clear spoken sentences. "
    "Prioritize in this exact order: (1) Immediate physical hazards or obstacles, "
    "(2) Main objects or people and their positions using clock-face directions, "
    "(3) Important readable signs or labels. "
    "Output plain spoken text only. Keep your response under 60 words."
)

QUESTION_PROMPT = (
    "You are a visual assistant for a person with low vision. "
    "Answer the user's question directly and concisely based strictly on the image. "
    "Read any requested numbers, dosages, expiry dates, or labels word-for-word. "
    "If the requested text or object is blurry, obscured, or cut off, "
    "do not guess — tell the user clearly how to adjust the camera. "
    "Output plain spoken text only and keep your answer under 40 words."
)

UNANSWERABLE_ANSWER = (
    "The image is unclear or cut off. Please adjust the camera angle and try again."
)

def clean_tts_text(text: str) -> str:
    """Strip Markdown so the answer is TTS-ready."""
    text = re.sub(r"#+\s*", "", text)
    text = re.sub(r"[\*_]+", "", text)
    text = re.sub(r"^\s*[-*•]\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"`+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


class AssistiveDataset(Dataset):
    """Multimodal dataset for VizWiz, TextVQA, and DocVQA."""

    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        # Load image
        image_path = item.get("image_path")
        if image_path and Path(image_path).exists():
            image = Image.open(image_path).convert("RGB")
        else:
            image = Image.new("RGB", (336, 336), color=(128, 128, 128))

        question = item.get("question", "").strip()
        answer   = item.get("answer", "").strip()

        # Unanswerable examples (VizWiz) → teach safe refusal
        if item.get("unanswerable") and not answer:
            answer = UNANSWERABLE_ANSWER

        # Dual-mode routing
        is_describe = not question or question.lower() in ["describe", "describe this photo."]
        system_prompt = DESCRIBE_PROMPT if is_describe else QUESTION_PROMPT
        user_text     = "Describe this photo." if is_describe else question

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": user_text},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": clean_tts_text(answer)}]},
        ]

        text = self.processor.apply_chat_template(messages, tokenize=False)
        inputs = self.processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            padding=True,
        )

        sample = {k: v[0] for k, v in inputs.items()}
        sample["labels"] = sample["input_ids"].clone()
        return sample


train_dataset = AssistiveDataset(samples, processor)
print(f"✅ Dataset ready — {len(train_dataset):,} samples")

# Quick sanity check on one sample
sample = train_dataset[0]
print(f"   input_ids shape : {sample['input_ids'].shape}")
print(f"   labels shape    : {sample['labels'].shape}")

## 6. Train

In [ ]:
from transformers import TrainingArguments, Trainer
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    bf16                        = True,
    logging_steps               = 10,
    save_strategy               = "epoch",
    save_total_limit            = 2,          # keep last 2 checkpoints only
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,      # required for multimodal inputs
    report_to                   = "none",     # set "wandb" if you have it
    seed                        = SEED,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
)

print(f"🚀 Starting training — {EPOCHS} epoch(s), {len(train_dataset):,} samples")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Steps per epoch    : {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print()

trainer.train()

## 7. Save checkpoint

In [ ]:
import json
from datetime import datetime

# Save LoRA adapter weights + processor config
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

# Save training metadata alongside the checkpoint
meta = {
    "model_id"       : MODEL_ID,
    "trained_at"     : datetime.utcnow().isoformat() + "Z",
    "epochs"         : EPOCHS,
    "total_samples"  : len(samples),
    "source_mix"     : dict(Counter(s["source"] for s in samples)),
    "lora_rank"      : LORA_RANK,
    "lora_alpha"     : LORA_ALPHA,
    "learning_rate"  : LEARNING_RATE,
    "batch_size"     : BATCH_SIZE,
    "grad_accum"     : GRAD_ACCUM,
}
with open(f"{OUTPUT_DIR}/training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"✅ Checkpoint saved to: {OUTPUT_DIR}")
print(f"   Files:")
for p in sorted(Path(OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f"     {p.name:40s}  {size_mb:6.1f} MB")

## 8. Test inference with the fine-tuned model
Quick smoke test — does the checkpoint answer correctly?

In [ ]:
from PIL import Image as PILImage
import requests
from io import BytesIO

# Load a sample image from the web (replace with a local path if preferred)
TEST_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
TEST_QUESTION  = "What do you see in this image?"

try:
    response = requests.get(TEST_IMAGE_URL, timeout=10)
    test_image = PILImage.open(BytesIO(response.content)).convert("RGB")
except Exception:
    # Fallback to a blank image if no internet
    test_image = PILImage.new("RGB", (336, 336), color=(100, 149, 237))
    TEST_QUESTION = "Describe this image."

model.eval()
with torch.no_grad():
    messages = [
        {"role": "system", "content": QUESTION_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": test_image},
            {"type": "text",  "text": TEST_QUESTION},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[test_image], return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.2,
        top_p=0.8,
        do_sample=True,
    )
    answer = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)

print(f"Question : {TEST_QUESTION}")
print(f"Answer   : {clean_tts_text(answer)}")

## 9. Use in AskSight backend

After downloading the checkpoint from Kaggle, point the backend at it:

```bash
# In backend/.env
ASKSIGHT_MODEL_CHECKPOINT="/absolute/path/to/checkpoints/asksight-v1"
ASK_TIMEOUT_SECONDS=30   # fine-tuned on GPU is fast — 30s is enough
```

The backend's `inference.py` auto-detects that the checkpoint directory contains  
`adapter_config.json` and loads it as a LoRA adapter on top of the base model.

```python
# This happens automatically in qwen3_vl_engine.py:
if (checkpoint / "adapter_config.json").is_file():
    base_model_id = PeftConfig.from_pretrained(checkpoint).base_model_name_or_path
    # loads Qwen3-VL-4B-Instruct + your LoRA on top
```

No code changes needed in the backend — just update `.env` and restart.

## 10. (Optional) Push to HuggingFace Hub

In [ ]:
# Uncomment and run if you want to push to HuggingFace Hub

# from huggingface_hub import notebook_login
# notebook_login()

# HF_REPO = "your-username/asksight-qwen3-vl-4b-qlora"
# model.push_to_hub(HF_REPO, private=True)
# processor.push_to_hub(HF_REPO, private=True)
# print(f"✅ Pushed to https://huggingface.co/{HF_REPO}")